**Understanding Unicode**

(a) `chr(0)` returns the null character

(b) `__repr__()` gives an escape sequence while `print` renders the character on screen

(c) when printed with `print`, null character displays nothing. In some situations python calls `__repr__()`

**Unicode Encodings**

(a) UTF-8 is most efficient for ASCII characters (comprising the majority of text) which occupy one byte per character. With -16 and -32 they occupy 2 and 4 bytes respectively.

(b) Incorrect since it assumes each character is one byte because `b` traverses one byte at a time. Any non-ASCII character breaks this function.

(c) `b'\x80\x80'`

**BPE Training on TinyStories**

(a) Training takes ~14s and ~1GB peak memory usage. Longest token is *accomplishment*.

(b) Pretokenization takes most time.

**BPE Training on OpenWebText**

(a) Training took ~15 minutes with peak memory usage of ~14GB. "ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ" was the longest token.

(b) Has web text (HTML, code) plus longer tokens because of larger vocabulary size.

**Experiments with Tokenizers**

(a) 4.15 for tiny stories and 4.69 for OWT

(b) Training tiny stories on OWT gives 3.19 compression ratio

(c) Current throughput is ~3-4 million tokens/second processed. At 3-4 bytes/token this is roughly 14 MB/s. 825GB / 14 MB/s gives roughly 16 hours for PILE dataset.

(d) UINT16 appropriate since no token integer value will exceed what can be stored

**Transformer LM FLOPs Accounting**

For each model, the components and their FLOPs are (`T` is sequence length):

- **Embedding lookup**: no FLOPs (just indexing)
- **Per layer:**
  - RMSNorm ×2: negligible
  - QKV projections: 3 × matmul of `(T, d_model) × (d_model, d_model)`
  - Attention scores: `(T, d_k) × (d_k, T)` per head, ×h heads
  - Attention weighted sum: `(T, T) × (T, d_k)` per head, ×h heads
  - Output projection: `(T, d_model) × (d_model, d_model)`
  - SwiGLU w1, w3: 2 × `(T, d_model) × (d_model, d_ff)`
  - SwiGLU w2: `(T, d_ff) × (d_ff, d_model)`
- **Final norm**: negligible
- **LM head**: `(T, d_model) × (d_model, vocab_size)`

Using the rule FLOPs = 2mnp for an (m×n)×(n×p) multiply:

| Component | FLOPs per layer |
|---|---|
| Q, K, V projections (×3) | 3 × 2·T·d_model·d_model = 6Td² |
| Attention scores | 2·T·d_k·T × h = 2T²·d_model |
| Attention weighted sum | 2·T·T·d_k × h = 2T²·d_model |
| Output projection | 2·T·d_model·d_model = 2Td² |
| SwiGLU w1, w3 (×2) | 2 × 2·T·d_model·d_ff = 4T·d_model·d_ff |
| SwiGLU w2 | 2·T·d_ff·d_model = 2T·d_model·d_ff |

So per layer: **8Td² + 4T²d + 6T·d·d_ff**

And total: **L × (8Td² + 4T²d + 6T·d·d_ff) + 2T·d·V** (LM head)

where d = d_model, T = context_length, L = num_layers, V = vocab_size.

---

**(a) GPT-2 XL Parameters & Memory**

| Component | Parameters |
|---|---|
| Token embedding | 50,257 × 1,600 = 80,411,200 |
| Per layer: QKV projections (×3) | 3 × 1,600 × 1,600 = 7,680,000 |
| Per layer: Output projection | 1,600 × 1,600 = 2,560,000 |
| Per layer: SwiGLU w1, w2, w3 (×3) | 3 × 1,600 × 4,288 = 20,582,400 |
| Per layer: RMSNorm ×2 (gain vectors) | 2 × 1,600 = 3,200 |
| ×48 layers | 48 × 30,825,600 = 1,479,628,800 |
| Final RMSNorm | 1,600 |
| LM head | 50,257 × 1,600 = 80,411,200 |
| **Total** | **≈ 1,640,452,800 (~1.64B)** |

At 4 bytes per float32 parameter: **~6.56 GB**

---

**(b) GPT-2 XL FLOPs (T=1024, d=1600, d_ff=4288, L=48, V=50257)**

Per layer:

| Operation | FLOPs |
|---|---|
| QKV projections | 6 × 1024 × 1600² = 15,728,640,000 |
| Attention scores | 2 × 1024² × 1600 = 3,355,443,200 |
| Attention weighted sum | 2 × 1024² × 1600 = 3,355,443,200 |
| Output projection | 2 × 1024 × 1600² = 5,242,880,000 |
| SwiGLU (w1+w3+w2) | 6 × 1024 × 1600 × 4288 = 42,467,328,000 |
| **Per layer total** | **~70.15B** |

×48 layers: **~3,367B FLOPs**

LM head: 2 × 1024 × 1600 × 50257 = **~164.7B FLOPs**

**Grand total: ~3,532B FLOPs**

---

**(c) Which parts dominate?**

The SwiGLU FFN dominates, accounting for roughly 60% of per-layer FLOPs, because d_ff ≈ 2.67 × d_model meaning the FFN matmuls are substantially larger than the attention projections. The LM head is also significant (~4.7% of total) due to the large vocabulary size.

---

**(d) Scaling across GPT-2 variants**

Using d_ff = nearest multiple of 64 to (8/3)×d_model:

| Model | L | d | d_ff | T | QKV+Out FLOPs | Attn FLOPs | FFN FLOPs | LM Head FLOPs | Total FLOPs |
|---|---|---|---|---|---|---|---|---|---|
| Small | 12 | 768 | 2048 | 1024 | ~57.9B | ~38.7B | ~110.1B | ~79.0B | **~285.7B** |
| Medium | 24 | 1024 | 2731 | 1024 | ~206.2B | ~103.1B | ~413.8B | ~105.5B | **~828.6B** |
| Large | 36 | 1280 | 3392 | 1024 | ~484.4B | ~193.3B | ~970.0B | ~131.8B | **~1,779.5B** |
| XL | 48 | 1600 | 4288 | 1024 | ~1,006.6B | ~322.1B | ~2,040.1B | ~164.7B | **~3,533.5B** |

As proportion of total:

| Model | QKV+Out | Attention | FFN | LM Head |
|---|---|---|---|---|
| Small | 20.3% | 13.5% | 38.5% | 27.7% |
| Medium | 24.9% | 12.4% | 49.9% | 12.7% |
| Large | 27.2% | 10.9% | 54.5% | 7.4% |
| XL | 28.5% | 9.1% | 57.8% | 4.7% |

As the model scales, the FFN takes up a proportionally larger share of FLOPs since it scales as L·d·d_ff ∝ d², while the LM head (fixed vocab size) and attention scores become proportionally smaller. The LM head drops from 27.7% in GPT-2 Small to just 4.7% in XL because it doesn't scale with depth.

---

**(e) GPT-2 XL with T=16,384**

New attention FLOPs per layer: 4 × 16384² × 1600 = **1,717.99B** (×48 = **~82,464B**)

New per-layer totals with T=16384 vs T=1024 (16× increase):
- QKV/output projections scale linearly with T: ×16
- **Attention scores/weighted sum scale as T²: ×256**
- FFN scales linearly with T: ×16

Total FLOPs ≈ **~138,569B** vs ~3,532B — roughly a **39× increase**.

The attention component explodes from ~9% to ~60% of total FLOPs, because it scales quadratically with context length while everything else scales linearly. This is precisely why long-context transformers are expensive and why techniques like sparse attention and FlashAttention exist — to tame this quadratic growth.

**Tuning the learning rate**

1e1: loss decreases slowly

1e2: loss decreases rapidly

1e3: loss explodes

**AdamW Resource Accounting**

**Setup**

Let B = batch_size, T = context_length, L = num_layers, d = d_model, V = vocab_size, d_ff = (8/3)d. All tensors are float32 = 4 bytes.

---

**(a) Memory Decomposition**

**Parameters**

| Component | Parameters |
|---|---|
| Token embedding | V·d |
| Per layer: QKV projections | 3·d² |
| Per layer: Output projection | d² |
| Per layer: SwiGLU (w1, w2, w3) | 3·d·d_ff = 8d² |
| Per layer: RMSNorm ×2 | 2d |
| Final RMSNorm | d |
| LM head | V·d |

Total parameters: **P = 2Vd + L(12d² + 2d) + d ≈ 2Vd + 12Ld²**

Memory: **4P bytes**

---

**Activations**

During the forward pass, we must store intermediate activations for the backward pass. Per transformer block:

| Activation | Shape | Elements |
|---|---|---|
| RMSNorm 1 input | (B, T, d) | BTd |
| Q, K, V projections | 3×(B, T, d) | 3BTd |
| QKᵀ scores | (B, h, T, T) | BhT² |
| Softmax output | (B, h, T, T) | BhT² |
| Weighted sum of V | (B, T, d) | BTd |
| Output projection | (B, T, d) | BTd |
| RMSNorm 2 input | (B, T, d) | BTd |
| SwiGLU w1 output (gate) | (B, T, d_ff) | (8/3)BTd |
| SwiGLU w3 output | (B, T, d_ff) | (8/3)BTd |
| SiLU output | (B, T, d_ff) | (8/3)BTd |
| Element-wise product | (B, T, d_ff) | (8/3)BTd |
| SwiGLU w2 output | (B, T, d) | BTd |

Per block total: **~(50/3)BTd + BhT²**

Over L blocks plus final RMSNorm (BTd), embedding lookup (BTd), and logits (BTV):

**Total activations ≈ L·((50/3)BTd + BhT²) + BTV + 2BTd**

Memory: **4 × (L·((50/3)BTd + BhT²) + BTV + 2BTd) bytes**

---

**Gradients**

Each parameter has a corresponding gradient of the same size:

**Memory: 4P bytes** (same as parameters)

---

**Optimizer State**

AdamW stores first moment m and second moment v for each parameter:

**Memory: 2 × 4P = 8P bytes**

---

**Total Memory**

$$\text{Total} = 4P + 4P + 8P + 4 \times \left(L\left(\frac{50}{3}BTd + BhT^2\right) + BTV + 2BTd\right)$$

$$= 16P + 4BTV + 4BTd\left(\frac{50L}{3} + 2\right) + 4BLhT^2$$

$$\approx 16(2Vd + 12Ld^2) + 4BTV + \frac{200}{3}BLTd + 4BLhT^2$$

---

**(b) GPT-2 XL Instantiation**

Parameters: V=50257, T=1024, L=48, d=1600, h=25, d_ff=4288

**Fixed memory (16P × 4 bytes):**

P ≈ 2×50257×1600 + 12×48×1600² = 160,820,800 + 1,474,560,000 ≈ 1,635,380,800

16 × 4 × 1,635,380,800 bytes ≈ **~104.7 GB**

**Batch-dependent memory:**

- Logits: 4 × B × 1024 × 50257 ≈ **~196.6 MB × B**
- Attention activations: 4 × B × 25 × 48 × 1024² ≈ **~4.8 GB × B**
- FFN + other activations: 4 × B × 1024 × 1600 × (50/3 × 48 + 2) ≈ **~5.1 GB × B**

Total batch-dependent: **≈ 10.1 GB × B**

So the expression is:

$$\text{Memory} \approx 10.1 \text{ GB} \times B + 104.7 \text{ GB}$$

For 80 GB: 10.1B + 104.7 ≤ 80 → B ≤ (80 - 104.7)/10.1 < 0

The fixed memory alone exceeds 80 GB — GPT-2 XL **cannot fit on a single 80 GB GPU** with float32 for all components. This is why in practice people use mixed precision (float16/bfloat16) and techniques like gradient checkpointing.

---

**(c) AdamW FLOPs per Step**

The AdamW update requires per parameter approximately:

- Moment updates (m, v): ~6 FLOPs
- Bias correction: ~3 FLOPs
- Weight decay: ~2 FLOPs
- Parameter update: ~4 FLOPs

So ~15 FLOPs per parameter, giving **~15P FLOPs** for the optimizer step, which is negligible compared to the forward/backward passes.

**Total FLOPs per step ≈ 3F_fwd + 15P ≈ 3F_fwd**

In general:

$$\text{FLOPs per step} \approx 3 \times (8LTd^2 + 4LT^2d + 6LTd \cdot d_{ff} + 2TVd)$$

---

**(d) Training Time for GPT-2 XL**

**FLOPs per step (single sequence, T=1024):**

- Forward: ~3,532B FLOPs
- Backward: ~7,064B FLOPs
- Total per sequence: ~10,596B FLOPs ≈ 1.06 × 10¹³ FLOPs

Scaled to batch size 1024:

$$\text{FLOPs per step} = 1024 \times 1.06 \times 10^{13} \approx 1.085 \times 10^{16} \text{ FLOPs}$$

**Hardware throughput at 50% MFU:**

$$495 \times 10^{12} \times 0.5 = 247.5 \times 10^{12} \text{ FLOP/s}$$

**Time per step:**

$$\frac{1.085 \times 10^{16}}{247.5 \times 10^{12}} \approx 43.8 \text{ seconds}$$

**Total training time:**

$$400{,}000 \times 43.8 = 17{,}520{,}000 \text{ seconds} \approx \textbf{4,867 hours} \approx \textbf{203 days}$$

This illustrates why large model training requires hundreds to thousands of GPUs running in parallel — training GPT-2 XL on a single H100 would take over 6 months.

**Tune the Learning Rate**

(a) Tried max/min values of 3e-4/3e-5 (validation loss, 2.060), 1e-3/1e-4 (1.858), 3e-3/3e-4 (1.854), 1e-2/1e-3 (unstable). As can be seen I increased learning rate until it became unstable. The loss of 1.854 associated with the 3e-3/3e-4 learning rate is within the expected 1.45-2.00 limits for Apple MPS hardware.

(b) With the higher learning rates (1e-3, 3e-3) the validation loss converges faster than the slower learning rate (3e-4) but if we try higher (1e-2), validation loss diverges rather than converging.

**Batch Size Variations**

Both 32 and 64 batch size work well giving small validation loss values of ~1.85-1.87. Really small batch size of 8 hits a loss floor of roughly 2.4 (noise exceeds gradient updates). Really large batch size of 256 takes really long time; likely having hit a memory bandwidth issue. Batch of 128 seems to decrease loss but needs more iterations to confirm what its lowest loss will be. Choosing batch of 32 going forwards.

**Generate Text**

Starting with prompt *Once upon a time* the following response is generated (`temperature` 0.8, `top-p` 0.95):

*, there was a little boy named Tim. Tim loved to learn new things. One day, Tim went to the park with his mom and dad. He saw a big tree and wanted to learn how to climb it.
As Tim started to climb, he saw a small bird. The bird was singing a beautiful song. Tim wanted to learn how to climb the tree. He asked the bird, "How can I learn to climb the tree?" The bird said, "This is a lot of fun!"
Tim started to climb the tree. He climbed higher and higher. When he reached the top, he saw the whole park. He saw a big tree and went to sit on a branch. He thought the tree would be a fun place to teach Tim how to climb.
As Tim climbed the tree, he met a little girl named Sue. Sue was very good at climbing trees. Sue asked Tim if he could help her climb the tree. Tim was happy to help Sue. They both climbed the tree together and had a great time.<|endoftext|>*

Temperature and top-p affect quality of output. For example, low temperature (e.g. 0.5) gives fluent, gramatically correct but repetitive and formulaic output. Temperature of 1.5 gives more creative output but which is sometimes gibberish. Top-p also affects the output in a similar fashion.

**Remove RMSNorm and Train**

When I remove RMSNorm, the loss explodes to NaN since activations grow uncontrollably through the model. Note, initially, the model activations are bounded because of the Glorot initialisation process for the parameters. Large learning rates move the parameters such that the activations become large and eventually NaN. Lower learning rates avoid loss becoming NaN but are not strong enough compared to noise; this means the losses plateau above their true minimum values.

**Implement Post-Norm**

The post-norm version trains similarly to the pre-norm baseline although pre-norm validation loss (1.85) slightly lower than post-norm (1.93).

**Implement NoPE**

Using NoPE increases validation loss from 1.85 to 1.97.

**SiLU vs SwiGLU**

SiLU does not seem to train badly, achieving validation loss of 1.89 (vs 1.85 for SwiGLU baseline).

**Experiment on OWT**

OWT training gives validation loss of 4.59, much higher than TinyStories loss of 1.85. This is expected, since OWT dataset is much more complex and varied. Therefore, the model struggles to replicate all the patterns present in the data.

Using the prompt *According to a new study,* with `temperature` 0.8 and `top-p` 0.95 gives the following. The text is clearly less fluent than TinyStories. The output is worse because the task space is much more diverse (research, news, history, sports, entertainment) and the each specific type of article is harder to replicate. 

"that found that over 400 children were killed and 21 were taken to hospital for 13 months.

On Tuesday, Nissma and Nissma have had their children, said the majority of the hospital had been informed of their lack of this.

The hearing was scheduled in the next week of the week, which will take place with the help of the case.

A leading medical examiner, the medical director of the Human Services Health Foundation, said the "under-emotional and public services" were now "not at all" in the medical industry.

In addition to the complex, the study was released from the Washington-based company in the United States of America, and said the process of dealing with abuse is not too harmful to the illness.

"This is one of the many things you think were all about," the research team wrote in the Sept. 16 meeting in advance of a hospital who was home to the hospital.

"It's about the 10,000 patients who came to hospital with the doctors who received help from a clinic and wanted to help prevent things from being professionals and the health services, in order to provide coverage and to help families with the treatment of their children."

The study revealed that the"